#🎯 Project: Dog vs Cat Image Generator & Real/Fake Checker
Objective:

Train a Deep Convolutional GAN (DCGAN) on the Kaggle Cats vs Dogs dataset to generate realistic dog/cat images and build a Streamlit app where users can upload an image and check if it’s real or generated (fake).

#🗂️ Project Structure

dog_cat_gan/

  ├── backend/

    │   ├── train_dcgan.py         # Train DCGAN

    │   └── generator.h5           # Saved generator model

    │   └── discriminator.h5       # Saved discriminator

├── frontend/

    │   └── app.py                 # Streamlit real/fake checker

    └── requirements.txt

#✅ Step 1: Download Dataset via kagglehub



In [1]:
import kagglehub

# Download the minicats-dogs dataset
path = kagglehub.dataset_download("aleemaparakatta/cats-and-dogs-mini-dataset")
print("Dataset path:", path)


100%|██████████| 21.9M/21.9M [00:00<00:00, 158MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/aleemaparakatta/cats-and-dogs-mini-dataset/versions/1


#✅ Step 2: Train a DCGAN (train_dcgan.py)

In [2]:
%%writefile train_dcgan.py

import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Dense, Flatten, Reshape, LeakyReLU, BatchNormalization
from tensorflow.keras import Sequential
from tensorflow.keras.optimizers import Adam
import numpy as np
import os
from glob import glob
from PIL import Image

import kagglehub

# Download the minicats-dogs dataset
path = kagglehub.dataset_download("aleemaparakatta/cats-and-dogs-mini-dataset")
print("Dataset path:", path)


# Hyperparameters
img_size = 64
latent_dim = 100
batch_size = 64
epochs = 5000

# Load images
def load_images(folder):
    paths = glob(os.path.join(folder, '*/*.jpg'))
    imgs = [np.array(Image.open(p).resize((img_size, img_size))) for p in paths]
    return np.array(imgs)

images = load_images(os.path.join(path, 'train'))
images = (images - 127.5) / 127.5  # Scale to [-1,1]
dataset = tf.data.Dataset.from_tensor_slices(images).shuffle(1000).batch(batch_size)

# Build Generator
def build_generator():
    model = Sequential([
        Dense(8*8*256, input_dim=latent_dim),
        Reshape((8,8,256)),
        BatchNormalization(), LeakyReLU(),
        Conv2DTranspose(128, (4,4), strides=(2,2), padding='same'),
        BatchNormalization(), LeakyReLU(),
        Conv2DTranspose(64, (4,4), strides=(2,2), padding='same'),
        BatchNormalization(), LeakyReLU(),
        Conv2DTranspose(3, (4,4), strides=(2,2), padding='same', activation='tanh')
    ])
    return model

# Build Discriminator
def build_discriminator():
    model = Sequential([
        tf.keras.Input(shape=(img_size,img_size,3)),
        Conv2D(64, (4,4), strides=(2,2), padding='same'),
        LeakyReLU(),
        Conv2D(128, (4,4), strides=(2,2), padding='same'),
        BatchNormalization(), LeakyReLU(),
        Flatten(),
        Dense(1, activation='sigmoid')
    ])
    return model

# Instantiate
generator = build_generator()
discriminator = build_discriminator()
discriminator.compile(optimizer=Adam(0.0002), loss='binary_crossentropy')

# GAN assembly
discriminator.trainable = False
gan = Sequential([generator, discriminator])
gan.compile(optimizer=Adam(0.0002), loss='binary_crossentropy')

# Training loop
for epoch in range(epochs):
    for real in dataset:
        noise = np.random.normal(0,1,(batch_size,latent_dim))
        fake = generator.predict(noise)
        d_loss = 0.5*(discriminator.train_on_batch(real, np.ones((len(real),1))) +
                      discriminator.train_on_batch(fake, np.zeros((batch_size,1))))
        g_loss = gan.train_on_batch(noise, np.ones((batch_size,1)))

    if epoch % 500 == 0:
        print(f"[Epoch {epoch}] D loss: {d_loss:.4f}, G loss: {g_loss:.4f}")

# Save models
generator.save('generator.h5')
discriminator.save('discriminator.h5')
print("✅ Models saved")


Writing train_dcgan.py


#✅ Step 3: Streamlit App – Frontend (app.py)

In [3]:

%%writefile dcganapp.py

import streamlit as st
from tensorflow.keras.models import load_model
from PIL import Image
import numpy as np

st.title("Dog/Cat Real-or-Fake Checker")
st.write("Upload a 64×64 image of a dog or cat to check if it's real or GAN-generated.")

disc = load_model("../generator.h5", compile=False)  # fix path

uploaded = st.file_uploader("Upload Image", type=["jpg","png"])
if uploaded:
    img = Image.open(uploaded).resize((64,64))
    st.image(img, caption="Uploaded Image", use_column_width=True)
    img_arr = np.array(img)/127.5 - 1
    img_arr = np.expand_dims(img_arr, axis=0)

    if st.button("Check"):
        score = disc.predict(img_arr)[0][0]
        label = "REAL" if score > 0.5 else "FAKE"
        st.write(f"Discriminator confidence: **{score:.2f}** → {label}")


Writing dcganapp.py


In [4]:
%%writefile requirements.txt
tensorflow>=2.15.0
streamlit>=1.35.0
kagglehub>=0.2.0
Pillow
numpy


Writing requirements.txt
